# Torn Helper ML — Manual per-stock Level 2 evaluation

Run this notebook for **one stock at a time**. Select a stock ID, connect to the PostgreSQL shard that owns it, and inspect matured live predictions from `stock_engine.ml_v2_checks`.

**Safety:** this notebook is read-only. It does not change predictions, models, calibration, or database records. The existing GitHub Actions worker already checks matured outcomes and performs scheduled calibration; this notebook provides a focused per-stock audit and downloadable report.

## 1. Install dependencies
Run this cell once each time you start a fresh Colab runtime.

In [ ]:
%pip -q install "psycopg[binary]" pandas ipywidgets

## 2. Choose one stock and connect
Set `STOCK_ID` manually. Stocks are assigned two per shard: stocks 1–2 use project 1, stocks 3–4 use project 2, and so on. Paste the PostgreSQL connection string for that stock's shard when prompted. Prefer a dedicated **read-only** database role. Do not paste credentials into a saved notebook.

In [ ]:
from getpass import getpass
import pandas as pd
import psycopg
from IPython.display import display

STOCK_ID = 1  # <-- Manually change this to the stock you want to inspect (1–35)
LOOKBACK_DAYS = 180

if not 1 <= int(STOCK_ID) <= 35:
    raise ValueError('STOCK_ID must be between 1 and 35')

PROJECT_NUMBER = (int(STOCK_ID) + 1) // 2
print(f'Stock {STOCK_ID} belongs to shard/project {PROJECT_NUMBER}.')
PG_DSN = getpass('PostgreSQL DSN for this shard (input hidden): ')

conn = psycopg.connect(PG_DSN, sslmode='require', connect_timeout=20, read_only=True)
with conn.cursor() as cur:
    cur.execute('''
        select project_number, enabled
        from stock_engine.settings
        where singleton
    ''')
    settings = cur.fetchone()
    if not settings or int(settings[0]) != PROJECT_NUMBER or not settings[1]:
        conn.close()
        raise RuntimeError('Shard assignment mismatch or shard is disabled.')
    cur.execute('select exists(select 1 from stock_engine.stocks where stock_id = %s)', (STOCK_ID,))
    if not cur.fetchone()[0]:
        conn.close()
        raise ValueError(f'Stock {STOCK_ID} is not present in this shard.')

print('Connected in read-only mode.')

## 3. Load matured prediction outcomes for this stock
Only completed outcomes are evaluated. The query is limited to the selected stock and the most recent `LOOKBACK_DAYS`.

In [ ]:
query = '''
select stock_id, style, analysis_minutes, horizon_minutes,
       origin_at, target_at, issued_at, checked_at,
       origin_price, raw_return, predicted_return, actual_return, model_id
from stock_engine.ml_v2_checks
where stock_id = %s
  and checked_at is not null
  and actual_return is not null
  and target_at <= now()
  and origin_at >= now() - (%s * interval '1 day')
order by target_at
'''
df = pd.read_sql_query(query, conn, params=(STOCK_ID, LOOKBACK_DAYS))
if df.empty:
    print(f'No matured prediction outcomes found for stock {STOCK_ID} in the last {LOOKBACK_DAYS} days.')
else:
    df['absolute_error'] = (df['actual_return'] - df['predicted_return']).abs()
    df['squared_error'] = (df['actual_return'] - df['predicted_return']) ** 2
    df['direction_correct'] = ((df['predicted_return'] >= 0) == (df['actual_return'] >= 0))
    print(f"Loaded {len(df):,} matured outcomes for stock {STOCK_ID}.")
    display(df.head())

## 4. Overall performance and per-configuration results
Returns and errors are shown as percentages. These are prediction-accuracy metrics, not trading-profit metrics; no fees or slippage are applied.

In [ ]:
def summarize(group):
    return pd.Series({
        'samples': len(group),
        'MAE_pct': group['absolute_error'].mean() * 100,
        'RMSE_pct': (group['squared_error'].mean() ** 0.5) * 100,
        'directional_accuracy_pct': group['direction_correct'].mean() * 100,
        'mean_predicted_return_pct': group['predicted_return'].mean() * 100,
        'mean_actual_return_pct': group['actual_return'].mean() * 100,
        'latest_target_at': group['target_at'].max(),
    })

if not df.empty:
    overall = summarize(df).to_frame('value')
    print('Overall results')
    display(overall)

    by_configuration = (
        df.groupby(['style', 'analysis_minutes', 'horizon_minutes'], dropna=False)
          .apply(summarize, include_groups=False)
          .reset_index()
          .sort_values(['MAE_pct', 'samples'], ascending=[True, False])
    )
    print('Results by analysis range and forecast horizon')
    display(by_configuration)

    by_model_id = (
        df.groupby(['style', 'analysis_minutes', 'horizon_minutes', 'model_id'], dropna=False)
          .apply(summarize, include_groups=False)
          .reset_index()
          .sort_values(['MAE_pct', 'samples'], ascending=[True, False])
    )
    print('Results by deployed model ID')
    display(by_model_id)

## 5. Compare recent and earlier performance
This is a diagnostic only. It flags configurations whose recent MAE is worse than their earlier MAE; it does not automatically change the selected model or calibration.

In [ ]:
if not df.empty:
    ordered = df.sort_values('target_at').copy()
    ordered['period'] = 'earlier'
    # Compare the latest half of observations with the preceding half, per selector.
    drift_rows = []
    keys = ['style', 'analysis_minutes', 'horizon_minutes']
    for key, group in ordered.groupby(keys, dropna=False):
        group = group.sort_values('target_at')
        if len(group) < 10:
            continue
        split = len(group) // 2
        earlier = group.iloc[:split]
        recent = group.iloc[split:]
        earlier_mae = earlier['absolute_error'].mean()
        recent_mae = recent['absolute_error'].mean()
        drift_rows.append({
            'style': key[0], 'analysis_minutes': key[1], 'horizon_minutes': key[2],
            'earlier_samples': len(earlier), 'recent_samples': len(recent),
            'earlier_MAE_pct': earlier_mae * 100,
            'recent_MAE_pct': recent_mae * 100,
            'MAE_change_pct': ((recent_mae / earlier_mae - 1) * 100) if earlier_mae > 0 else None,
            'diagnostic': 'recent MAE higher' if recent_mae > earlier_mae else 'no MAE deterioration detected',
        })
    drift = pd.DataFrame(drift_rows)
    display(drift.sort_values('MAE_change_pct', ascending=False) if not drift.empty else drift)
else:
    drift = pd.DataFrame()

## 6. Export this stock's report
The CSV files are analysis outputs only. They do not modify the database or deploy a model.

In [ ]:
from pathlib import Path

if not df.empty:
    prefix = f'stock_{STOCK_ID}_level2'
    df.to_csv(f'{prefix}_matured_predictions.csv', index=False)
    by_configuration.to_csv(f'{prefix}_configuration_summary.csv', index=False)
    by_model_id.to_csv(f'{prefix}_model_summary.csv', index=False)
    drift.to_csv(f'{prefix}_drift_diagnostic.csv', index=False)
    print('Saved report CSV files:', prefix)
else:
    print('Nothing to export yet.')

conn.close()
print('Database connection closed.')

## Important interpretation notes
- This notebook evaluates only forecasts already recorded in `ml_v2_checks`; it does not generate new minute-by-minute forecasts.
- The production worker currently checks matured outcomes and performs scheduled calibration. This notebook does not duplicate or override that process.
- Model IDs identify the model that issued each forecast; they are not hyperparameter-search trial IDs.
- Small sample counts can make MAE and directional accuracy unstable. Treat the drift table as a review signal, not an automatic promotion decision.
- To inspect another stock, change `STOCK_ID`, connect to its assigned shard, and rerun the notebook from the top.